In [1]:
import pysam
import pandas as pd
import numpy as np
import collections
import os
from Bio.Seq import Seq
import ast

In [2]:
import scipy
import networkx as nx

In [3]:
print(scipy.__version__)
print(nx.__version__)

1.16.1
3.5


In [2]:
print(pysam.__version__)

0.23.3


In [3]:
hprcDF = pd.read_csv('/LeeLab/HPRC/chromosomeY/Data/AmpliconicGenes/HPRC_chrYAmpliconicGene_Filtered_wLiftOffAnnotation_06162025.csv')
hgsvcDF = pd.read_csv('/LeeLab/HPRC/chromosomeY/Data/AmpliconicGenes/HGSVC3_chrYAmpliconicGene_Filtered_wLiftOffAnnotation_07162025.csv')
cephDF = pd.read_csv('/LeeLab/HPRC/chromosomeY/Data/AmpliconicGenes/CEPH_chrYAmpliconicGene_Filtered_wLiftOffAnnotation_09092025.csv')


fec_hprcDF = hprcDF.copy()#[hprcDF['geneCompletenessInfo']=='Full_Exon_Copy'].copy()
fec_hgsvcDF = hgsvcDF.copy()#[hgsvcDF['geneCompletenessInfo']=='Full_Exon_Copy'].copy()
fec_cephDF = cephDF.copy()#[cephDF['geneCompletenessInfo']=='Full_Exon_Copy'].copy()

combinedDF = pd.concat([fec_hprcDF,fec_hgsvcDF,fec_cephDF])
combinedDF.reset_index(inplace=True)

In [4]:
counts=[]
for sample in set(combinedDF['sampleName']):
    testDF = combinedDF[combinedDF['sampleName']==sample].copy()
    tempDict = collections.Counter(testDF['geneName'])
    #print(tempDict)
    for gene in tempDict.keys():
        geneDF = testDF[testDF['geneName']==gene].copy()

        counts.append([sample, gene, tempDict[gene], len(set(geneDF['contig']))])
countDF = pd.DataFrame(data=counts, columns=['Sample','Gene','FullExonCopies','Contigs'])

In [5]:
countDF

,Sample,Gene,FullExonCopies,Contigs
0,HG02071,PRY2,6,1
1,HG02071,TSPY1,36,1
2,HG02071,VCY,2,1
3,HG02071,XKRY,2,1
4,HG02071,CDY2,2,1
...,...,...,...,...
1388,HG03065,CDY1,5,1
1389,HG03065,HSFY1,2,1
1390,HG03065,RBMY1B,9,1
1391,HG03065,BPY2,4,1


In [6]:
assemblyDict={}
directory='/LeeLab/Assemblies/HPRC_Release2/chrY_assemblies/'
for file in os.listdir(directory):
    if '.fai' in file or '.gzi' in file or '.DS' in file:
        continue
    else:
        assemblyDict[file.split("_")[0]]= directory+file
print(len(assemblyDict))

143


In [7]:
tempLines=[]
with open('/LeeLab/HPRC/chromosomeY/Data/HMMER_Files/HG02647.b0c803cf.DYZ19_Yq.hmmer-tblout.txt') as file:
    for line in file:
        if '#' in str(line):
            continue
        else:
            tempLines.append(line.split())
tempDF = pd.DataFrame(data=tempLines)
tempDF.sort_values(by=[0,6], inplace=True)
tempDF[12]=tempDF[12].astype(float)

In [8]:
flag=0
tempDistances= []
for row in tempDF.index:
    if flag==0:
        previous = int(tempDF.at[row,8])
        tempDistances.append(0)
        flag+=1
    else:
        tempDistances.append(int(tempDF.at[row,9])-previous)
        previous = int(tempDF.at[row,8])
tempDF['DBH']=tempDistances

In [9]:
collections.Counter(tempDF.reset_index().iloc[23:2185]['DBH'])

Counter({1: 2113, 2: 38, 0: 8, 101: 1, -2: 1, -1: 1})

In [10]:
tempDF.reset_index(inplace=True)

In [11]:
tempDF.iloc[280:285]

,index,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,DBH
280,503,HG02647_chrY,-,DYZ19_Yq,-,3,124,23306729,23306608,23306731,23306607,54649369,-,1.700000e-28,102.9,2.7,-,1
281,1523,HG02647_chrY,-,DYZ19_Yq,-,3,124,23306854,23306733,23306856,23306732,54649369,-,4.000000e-26,95.3,7.6,-,1
282,729,HG02647_chrY,-,DYZ19_Yq,-,3,123,23306979,23306859,23306981,23306857,54649369,-,5.500000e-28,101.2,4.0,-,1
283,810,HG02647_chrY,-,DYZ19_Yq,-,4,124,23307203,23307083,23307206,23307082,54649369,-,8.000000e-28,100.7,4.0,-,101
284,1312,HG02647_chrY,-,DYZ19_Yq,-,3,124,23307329,23307208,23307331,23307207,54649369,-,1.200000e-26,97.0,5.6,-,1


In [ ]:
HG00512_chrY:26111279-26132482_+_FEC_9

In [13]:
import pysam
#print(pysam.faidx(assemblyDict['HG00512'], 'HG00512_chrY:26111279-26132482'))
#HG00512,V2_Assembly,HG00512_chrY,BPY2,Full_Exon_Copy,9,0.0888888888888888,26111279,26132482,+,"
#[['290', '0.0', '0.0', '0.0', 'HG00512_chrY', 26111279, 26111309, '(55608585)', '+', 'BPY2_EXON1', 'Unspecified', '1', '31', '(0)', '15719', 'HG00512', 'V2_Assembly', 'BPY2_Copy_4', 'Full_Exon_Copy', 'BPY2', 0.08888888888888889], 
# ['1149', '0.0', '0.0', '0.0', 'HG00512_chrY', 26111585, 26111707, '(55608187)', '+', 'BPY2_EXON2', 'Unspecified', '1', '123', '(0)', '15720', 'HG00512', 'V2_Assembly', 'BPY2_Copy_4', 'Full_Exon_Copy', 'BPY2', 0.08888888888888889], ['1281', '0.0', '0.0', '0.0', 'HG00512_chrY', 26114773, 26114909, '(55604985)', '+', 'BPY2_EXON3', 'Unspecified', '1', '137', '(0)', '15729', 'HG00512', 'V2_Assembly', 'BPY2_Copy_4', 'Full_Exon_Copy', 'BPY2', 0.08888888888888889], 
# ['1112', '0.0', '0.0', '0.0', 'HG00512_chrY', 26119319, 26119437, '(55600457)', '+', 'BPY2_EXON4', 'Unspecified', '1', '119', '(0)', '15737', 'HG00512', 'V2_Assembly', 'BPY2_Copy_4', 'Full_Exon_Copy', 'BPY2', 0.08888888888888889], ['1100', '0.0', '0.0', '0.0', 'HG00512_chrY', 26121497, 26121614, '(55598280)', '+', 'BPY2_EXON5', 'Unspecified', '1', '118', '(0)', '15741', 'HG00512', 'V2_Assembly', 'BPY2_Copy_4', 'Full_Exon_Copy', 'BPY2', 0.08888888888888889], ['994', '0.0', '0.0', '0.0', 'HG00512_chrY', 26124469, 26124574, '(55595320)', '+', 'BPY2_EXON6', 'Unspecified', '1', '106', '(0)', '15750', 'HG00512', 'V2_Assembly', 'BPY2_Copy_4', 'Full_Exon_Copy', 'BPY2', 0.08888888888888889], ['1158', '0.8', '0.0', '0.0', 'HG00512_chrY', 26125267, 26125391, '(55594503)', '+', 'BPY2_EXON7', 'Unspecified', '1', '125', '(0)', '15753', 'HG00512', 'V2_Assembly', 'BPY2_Copy_4', 'Full_Exon_Copy', 'BPY2', 0.08888888888888889], ['1171', '0.0', '0.0', '0.0', 'HG00512_chrY', 26131942, 26132066, '(55587828)', '+', 'BPY2_EXON8', 'Unspecified', '1', '125', '(0)', '15764', 'HG00512', 'V2_Assembly', 'BPY2_Copy_4', 'Full_Exon_Copy', 'BPY2', 0.08888888888888889], ['2986', '0.0', '0.0', '0.0', 'HG00512_chrY', 26132161, 26132482, '(55587412)', '+', 'BPY2_EXON9', 'Unspecified', '1', '322', '(0)', '15765', 'HG00512', 'V2_Assembly', 'BPY2_Copy_4', 'Full_Exon_Copy', 'BPY2', 0.08888888888888889]]",2,BPY2,HG00512_chrY:26111279-26132482,BPY2,HG00512_chrY:26111279-26132482,"['HG00512_chrY:26111279-26111309', 'HG00512_chrY:26111585-26111707', 'HG00512_chrY:26114773-26114909', 'HG00512_chrY:26119319-26119437', 'HG00512_chrY:26121497-26121614', 'HG00512_chrY:26124469-26124574', 'HG00512_chrY:26125267-26125391', 'HG00512_chrY:26131942-26132066', 'HG00512_chrY:26132161-26132482']"


In [33]:
#tempDF.to_csv("/HG02647.csv")

In [13]:
#this code pulls the entire gene with introns
folderDirectory='/LeeLab/HPRC/chromosomeY/Data/AmpliconicGenes/PulledSequences/FullGenewIntrons/'
#for sample in set(combinedDF['sampleName']):
    testDF = combinedDF[combinedDF['sampleName']==sample].copy()
    tempDict = collections.Counter(testDF['geneName'])
    for gene in tempDict.keys():
        geneDF = testDF[testDF['geneName']==gene].copy()

        with open(folderDirectory+"HPRC-HGSVC3-CEPH_v2_"+gene+".wIntrons.fasta", 'a+') as file:
            for row in geneDF.index:

                mySeqName = str(geneDF.at[row,'contig'])+":"+str(geneDF.at[row,'contigStart'])+"-"+str(geneDF.at[row,'contigEnd'])
                #mySeqName2 = str(geneDF.at[row,'contig'])+":"+str(int(geneDF.at[row,'contigStart'])-500)+"-"+str(int(geneDF.at[row,'contigEnd'])+500)
                if str(geneDF.at[row,'orientation']) == '+':
                    sequencePull = (pysam.faidx(assemblyDict[sample],mySeqName))
                    sequence = ''.join(sequencePull.split()[1:])
                    
                else:
                    sequencePull = (pysam.faidx(assemblyDict[sample], mySeqName))
                    sequence = str(Seq(''.join(sequencePull.split()[1:])).reverse_complement())


                if str(geneDF.at[row,'geneCompletenessInfo'])=='Full_Exon_Copy':
                    file.write(">"+mySeqName+"_"+str(geneDF.at[row,'orientation'])+"_FEC_"+str(str(geneDF.at[row,'totalExons']))+"\n")
                    file.write(sequence+"\n")
                else:
                    file.write(">"+mySeqName+"_"+str(geneDF.at[row,'orientation'])+"_TEC_"+str(str(geneDF.at[row,'totalExons']))+"\n")
                    file.write(sequence+"\n")

        file.close()

In [19]:
#this code pulls the entire DAZ gene with introns + 500 bases
folderDirectory='/LeeLab/HPRC/chromosomeY/Data/AmpliconicGenes/PulledSequences/DAZMappings/'
#for sample in set(combinedDF['sampleName']):
    testDF = combinedDF[combinedDF['sampleName']==sample].copy()
    tempDict = collections.Counter(testDF['geneName'])
    for gene in ['DAZ1']:
        geneDF = testDF[testDF['geneName']==gene].copy()

        for row in geneDF.index:
            
            mySeqName = str(geneDF.at[row,'contig'])+":"+str(geneDF.at[row,'contigStart'])+"-"+str(geneDF.at[row,'contigEnd'])
            if int(geneDF.at[row,'contigStart'])>500:
                mySeqName2 = str(geneDF.at[row,'contig'])+":"+str(int(geneDF.at[row,'contigStart'])-500)+"-"+str(int(geneDF.at[row,'contigEnd'])+500)

            else:
                minimum = int(geneDF.at[row,'contigStart'])-1
                mySeqName2 = str(geneDF.at[row,'contig'])+":"+str(int(geneDF.at[row,'contigStart'])-minimum)+"-"+str(int(geneDF.at[row,'contigEnd'])+500)

                
            with open(folderDirectory+mySeqName.replace(":","_")+".fasta", 'a+') as file:
                
                sequencePull = (pysam.faidx(assemblyDict[sample],mySeqName2))
                sequence = ''.join(sequencePull.split()[1:])

                if str(geneDF.at[row,'geneCompletenessInfo'])=='Full_Exon_Copy':
                    file.write(">"+mySeqName+"\n")
                    file.write(sequence+"\n")
                else:
                    file.write(">"+mySeqName+"\n")
                    file.write(sequence+"\n")

            file.close()

In [14]:
#this code pulls only the exons of the gene and stitches them back together
folderDirectory='/LeeLab/HPRC/chromosomeY/Data/AmpliconicGenes/PulledSequences/StitchedExons/'
#for sample in set(combinedDF['sampleName']):
    testDF = combinedDF[combinedDF['sampleName']==sample].copy()
    tempDict = collections.Counter(testDF['geneName'])
    for gene in tempDict.keys():
        geneDF = testDF[testDF['geneName']==gene].copy()

        with open(folderDirectory+"HPRC-HGSVC3-CEPH_v2_"+gene+".Exons.fasta", 'a+') as file:
            for row in geneDF.index:
    
                mySeqName = str(geneDF.at[row,'contig'])+":"+str(geneDF.at[row,'contigStart'])+"-"+str(geneDF.at[row,'contigEnd'])
                exonList = ast.literal_eval(geneDF.at[row,'Exons'])
    
                sequences=[]
                if str(geneDF.at[row,'orientation']) == '+':
                    for exon in exonList:
                        if exon[11]=='1' and exon[13]=='(0)':
                            sequencePull = ''.join((pysam.faidx(assemblyDict[sample], str(exon[4])+":"+str(exon[5])+"-"+str(exon[6]))).split()[1:])
                            sequences.append(sequencePull)
                        else:
    
                            firstNumber = int(exon[11])-1
                            secondNumber = int(exon[13].split("(")[1].split(")")[0])
                            if firstNumber <10 and secondNumber < 10:
                                sequencePull = ''.join(pysam.faidx(assemblyDict[sample], str(exon[4])+":"+str(int(exon[5])-firstNumber)+"-"+str(int(exon[6])+secondNumber)).split()[1:])
                                sequences.append(sequencePull)
                            else:
                                sequencePull = ''.join((pysam.faidx(assemblyDict[sample], str(exon[4])+":"+str(exon[5])+"-"+str(exon[6]))).split()[1:])
                                sequences.append(sequencePull)
                    
                    sequence = ''.join(sequences)
                    
                else:
                    
                    for exon in exonList:
                        if exon[11]=='(0)' and exon[13]=='1':
                            sequencePull = ''.join((pysam.faidx(assemblyDict[sample], str(exon[4])+":"+str(exon[5])+"-"+str(exon[6]))).split()[1:])
                            sequences.append(sequencePull)
                        else:
                            firstNumber = int(exon[11].split("(")[1].split(")")[0])
                            secondNumber = int(exon[13])-1
                            if firstNumber <10 and secondNumber < 10:
                                sequencePull = ''.join(pysam.faidx(assemblyDict[sample], str(exon[4])+":"+str(int(exon[5])-firstNumber)+"-"+str(int(exon[6])+secondNumber)).split()[1:])
                                sequences.append(sequencePull)
                            else:
                                sequencePull = ''.join((pysam.faidx(assemblyDict[sample], str(exon[4])+":"+str(exon[5])+"-"+str(exon[6]))).split()[1:])
                                sequences.append(sequencePull)
                            
                    
                    sequence = str(Seq(''.join(sequences)).reverse_complement())
    
    
                if str(geneDF.at[row,'geneCompletenessInfo'])=='Full_Exon_Copy':
                    file.write(">"+mySeqName+"_"+str(geneDF.at[row,'orientation'])+"_FEC_"+str(str(geneDF.at[row,'totalExons']))+"\n")
                    file.write(sequence+"\n")
                else:
                    file.write(">"+mySeqName+"_"+str(geneDF.at[row,'orientation'])+"_TEC_"+str(str(geneDF.at[row,'totalExons']))+"\n")
                    file.write(sequence+"\n")

        file.close()